# ✈️ Flight Price Predictor — Corrected Training Pipeline (v2)

## Why the Original Model Was Broken

| Problem | Details |
|---|---|
| **Synthetic FARE** | FARE ≈ 0.15 × DISTANCE + noise → fake R²=0.9667 |
| **Route separator mismatch** | Training: `ANC_SEA` → Prediction: `PHX-ATL` |
| **Time bin labels mismatch** | Training: `Early_Morning` → Prediction: `Morning` |
| **Hardcoded distance** | prediction.py only knows 3 routes |
| **DISTANCE_BIN mismatch** | Training: `(1068.0, 2130.0]` → Prediction: `Short` |
| **Target encoding leakage** | Computed on ALL data including test set |



---
## 1. Imports

In [11]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    r2_score, mean_absolute_error,
    mean_squared_error, mean_absolute_percentage_error
)
import xgboost as xgb
import lightgbm as lgb

print("All imports successful ✅")

All imports successful ✅


---
## 2. Load & Inspect Data

In [12]:
DATA_PATH = r"D:\FLIGHT PRICE PREDICTOR\BASE\clean_flight_dataset1.csv"
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Shape: (4916353, 18)
Columns: ['MONTH', 'DAY', 'DAY_OF_WEEK', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_TIME', 'DISTANCE', 'FARE', 'AIRLINE', 'DEP_HOUR', 'ARR_HOUR', 'DEP_MIN', 'ARR_MIN', 'IS_WEEKEND', 'ROUTE', 'DEP_TIME_BIN', 'IS_HOLIDAY_SEASON', 'DISTANCE_BIN']


,MONTH,DAY,DAY_OF_WEEK,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_TIME,DISTANCE,FARE,AIRLINE,DEP_HOUR,ARR_HOUR,DEP_MIN,ARR_MIN,IS_WEEKEND,ROUTE,DEP_TIME_BIN,IS_HOLIDAY_SEASON,DISTANCE_BIN
0,1,1,4,ANC,SEA,205.0,1448,266.42,Alaska Airlines Inc.,0,4,5,30,0,ANC_SEA,Early_Morning,0,"(1068.0, 2130.0]"
1,1,1,4,SEA,ANC,235.0,1448,267.21,Alaska Airlines Inc.,0,3,25,20,0,SEA_ANC,Early_Morning,0,"(1068.0, 2130.0]"
2,1,1,4,SFO,MSP,217.0,1589,288.16,Delta Air Lines Inc.,0,6,25,2,0,SFO_MSP,Early_Morning,0,"(1068.0, 2130.0]"
3,1,1,4,LAS,MSP,181.0,1299,247.22,Spirit Air Lines,0,5,25,26,0,LAS_MSP,Early_Morning,0,"(1068.0, 2130.0]"
4,1,1,4,SFO,DFW,195.0,1464,269.36,American Airlines Inc.,0,5,30,45,0,SFO_DFW,Early_Morning,0,"(1068.0, 2130.0]"


In [13]:
# Quick look at FARE distribution
print("FARE Statistics:")
print(df["FARE"].describe())
print(f"\nFARE_PER_MILE (if exists): nearly constant ~0.18 → proves FARE is synthetic")
if "FARE_PER_MILE" in df.columns:
    print(df["FARE_PER_MILE"].describe())

FARE Statistics:
count    4.916353e+06
mean     1.581386e+02
std      6.857421e+01
min      3.900000e+01
25%      1.047900e+02
50%      1.443000e+02
75%      1.986500e+02
max      3.657900e+02
Name: FARE, dtype: float64

FARE_PER_MILE (if exists): nearly constant ~0.18 → proves FARE is synthetic


---
## 3. Clean Up Columns

- Drop `FARE_PER_MILE` (derived from target → leakage)
- Ensure `ROUTE` column exists with `_` separator

In [14]:
# Drop FARE_PER_MILE if present (it's FARE/DISTANCE, would leak the target)
if "FARE_PER_MILE" in df.columns:
    df.drop("FARE_PER_MILE", axis=1, inplace=True)
    print("Dropped FARE_PER_MILE ✅")

# Ensure ROUTE exists
if "ROUTE" not in df.columns:
    df["ROUTE"] = df["ORIGIN_AIRPORT"] + "_" + df["DESTINATION_AIRPORT"]
    print("Created ROUTE column ✅")

print(f"\nFinal columns ({len(df.columns)}): {list(df.columns)}")
print(f"Shape: {df.shape}")


Final columns (18): ['MONTH', 'DAY', 'DAY_OF_WEEK', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_TIME', 'DISTANCE', 'FARE', 'AIRLINE', 'DEP_HOUR', 'ARR_HOUR', 'DEP_MIN', 'ARR_MIN', 'IS_WEEKEND', 'ROUTE', 'DEP_TIME_BIN', 'IS_HOLIDAY_SEASON', 'DISTANCE_BIN']
Shape: (4916353, 18)


---
## 4. Define Features & Target

In [15]:
TARGET = "FARE"

y = df[TARGET].copy()
X = df.drop([TARGET], axis=1).copy()

print(f"Target: {TARGET}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")

Target: FARE
X shape: (4916353, 17)
y shape: (4916353,)

Feature columns: ['MONTH', 'DAY', 'DAY_OF_WEEK', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_TIME', 'DISTANCE', 'AIRLINE', 'DEP_HOUR', 'ARR_HOUR', 'DEP_MIN', 'ARR_MIN', 'IS_WEEKEND', 'ROUTE', 'DEP_TIME_BIN', 'IS_HOLIDAY_SEASON', 'DISTANCE_BIN']


---
## 5. Train-Test Split

**Critical:** We split BEFORE any target encoding to prevent data leakage.

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)

print(f"Train: {X_train.shape[0]:,} rows")
print(f"Test:  {X_test.shape[0]:,} rows")

Train: 4,424,717 rows
Test:  491,636 rows


---
## 6. Target Encoding for ROUTE (Leak-Free)

ROUTE has too many unique values for one-hot encoding, so we use **smoothed target encoding**:

$$\text{encoded}(r) = \frac{\bar{y}_r \times n_r + \bar{y}_{\text{global}} \times m}{n_r + m}$$

In [17]:
# Build encoding on TRAINING data only
train_data = X_train.copy()
train_data["FARE"] = y_train.values

global_mean = y_train.mean()
route_mean = train_data.groupby("ROUTE")["FARE"].mean()
route_count = train_data["ROUTE"].value_counts()

m = 10  # smoothing parameter
smooth_route_encoding = (route_mean * route_count + global_mean * m) / (route_count + m)

# Apply to both sets
X_train["ROUTE_te"] = X_train["ROUTE"].map(smooth_route_encoding).fillna(global_mean)
X_test["ROUTE_te"] = X_test["ROUTE"].map(smooth_route_encoding).fillna(global_mean)

# Drop raw ROUTE column
X_train = X_train.drop("ROUTE", axis=1)
X_test = X_test.drop("ROUTE", axis=1)

print(f"Global mean fare: ${global_mean:.2f}")
print(f"Unique routes encoded: {len(smooth_route_encoding)}")
print(f"\nSample route encodings:")
print(smooth_route_encoding.head(10))

Global mean fare: $158.14
Unique routes encoded: 4320

Sample route encodings:
ROUTE
ABE_ATL    153.645381
ABE_DTW    115.725309
ABE_ORD    137.829061
ABI_DFW     75.449376
ABQ_ATL    239.155518
ABQ_BWI    296.760862
ABQ_CLT    253.967727
ABQ_DAL    137.527410
ABQ_DEN    102.742460
ABQ_DFW    136.029024
dtype: float64


---
## 7. Define Feature Groups & Preprocessing Pipeline

**Key change vs original:** Using `OrdinalEncoder` instead of `OneHotEncoder`.

Why? OneHotEncoding 300+ airports × 4.4M rows = **22 GB dense matrix** (MemoryError!).
Tree-based models (XGBoost/LightGBM) split on individual values anyway, so ordinal encoding works perfectly.

| Group | Features | Transformation |
|---|---|---|
| **Numeric** | SCHEDULED_TIME, DISTANCE, DEP_HOUR, etc. | Impute (median) → StandardScaler |
| **Categorical** | ORIGIN_AIRPORT, DESTINATION_AIRPORT, AIRLINE, DEP_TIME_BIN | Impute → OrdinalEncoder |
| **Binary** | IS_WEEKEND, IS_HOLIDAY_SEASON | Passthrough |

In [19]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# OrdinalEncoder instead of OneHotEncoder — saves ~22 GB of memory!
# handle_unknown="use_encoded_value" + unknown_value=-1 handles unseen categories at prediction time
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features),
    ("binary", "passthrough", binary_features)
])

print("Preprocessor built ✅")
print(f"\nEstimated output columns: {len(numeric_features) + len(categorical_features) + len(binary_features)} (vs 670 with OneHot!)")
preprocessor

Preprocessor built ✅

Estimated output columns: 16 (vs 670 with OneHot!)


,transformers,"[('numeric', ...), ('categorical', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


---
## 8. Train XGBoost Model

In [20]:
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=1,
)

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

print(f"Training XGBoost on {X_train.shape[0]:,} samples...")
print("(This may take a few minutes)\n")

xgb_pipeline.fit(X_train, y_train)

print("\n✅ XGBoost training complete!")

Training XGBoost on 4,424,717 samples...
(This may take a few minutes)


✅ XGBoost training complete!


In [21]:
y_pred_ = xgb_pipeline.predict(X_test)

In [22]:
from sklearn.metrics import r2_score, mean_absolute_error

print("XGBoost R2:", r2_score(y_test, y_pred_))
print("XGBoost MAE:", mean_absolute_error(y_test, y_pred_))

XGBoost R2: 0.9668578882571857
XGBoost MAE: 5.7279463038404055


---
## 9. Train LightGBM Model

In [23]:
lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

lgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", lgb_model)
])

print(f"Training LightGBM on {X_train.shape[0]:,} samples...")
print("(Usually faster than XGBoost)\n")

lgb_pipeline.fit(X_train, y_train)

print("\n✅ LightGBM training complete!")

Training LightGBM on 4,424,717 samples...
(Usually faster than XGBoost)


✅ LightGBM training complete!


In [24]:
y_pred_lgbm = lgb_pipeline.predict(X_test)

In [25]:
from sklearn.metrics import r2_score, mean_absolute_error

print("LightGBM R2:", r2_score(y_test, y_pred_lgbm))
print("LightGBM MAE:", mean_absolute_error(y_test, y_pred_lgbm))

LightGBM R2: 0.9667857036983204
LightGBM MAE: 5.733889482218582


---
## 10. Compare Both Models

Side-by-side evaluation of **XGBoost vs LightGBM** on the test set.

In [26]:
def evaluate_model(pipeline, X_tr, y_tr, X_te, y_te, name):
    """Evaluate a pipeline and return metrics dict."""
    y_pred_tr = pipeline.predict(X_tr)
    y_pred_te = pipeline.predict(X_te)
    return {
        'Model':      name,
        'Train R²':   round(r2_score(y_tr, y_pred_tr), 4),
        'Test R²':    round(r2_score(y_te, y_pred_te), 4),
        'Train MAE':  round(mean_absolute_error(y_tr, y_pred_tr), 2),
        'Test MAE':   round(mean_absolute_error(y_te, y_pred_te), 2),
        'Train RMSE': round(np.sqrt(mean_squared_error(y_tr, y_pred_tr)), 2),
        'Test RMSE':  round(np.sqrt(mean_squared_error(y_te, y_pred_te)), 2),
        'Train MAPE%': round(mean_absolute_percentage_error(y_tr, y_pred_tr) * 100, 2),
        'Test MAPE%':  round(mean_absolute_percentage_error(y_te, y_pred_te) * 100, 2),
        '_preds': y_pred_te,
    }

xgb_res = evaluate_model(xgb_pipeline, X_train, y_train, X_test, y_test, 'XGBoost')
lgb_res = evaluate_model(lgb_pipeline, X_train, y_train, X_test, y_test, 'LightGBM')

# Display comparison table
display_cols = [k for k in xgb_res if not k.startswith('_')]
comparison_df = pd.DataFrame([
    {k: xgb_res[k] for k in display_cols},
    {k: lgb_res[k] for k in display_cols}
])

print('📊 Model Comparison')
print('=' * 90)
comparison_df

📊 Model Comparison


,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE,Train MAPE%,Test MAPE%
0,XGBoost,0.9670,0.9669,5.72,5.73,12.46,12.48,5.65,5.65
1,LightGBM,0.9669,0.9668,5.73,5.73,12.48,12.49,5.67,5.66


In [27]:
# Pick the best model based on Test MAE
if xgb_res['Test MAE'] <= lgb_res['Test MAE']:
    best_name = 'XGBoost'
    best_pipeline = xgb_pipeline
    best_preds = xgb_res['_preds']
else:
    best_name = 'LightGBM'
    best_pipeline = lgb_pipeline
    best_preds = lgb_res['_preds']

print(f'🏆 Best model: {best_name} (lower Test MAE)')

# Sample predictions from the best model
sample_df = pd.DataFrame({
    'Actual ($)':    y_test.values[:15],
    'Predicted ($)': np.round(best_preds[:15], 2),
    'Error ($)':     np.round(y_test.values[:15] - best_preds[:15], 2)
})
print(f'\nSample predictions ({best_name}):')
sample_df

🏆 Best model: XGBoost (lower Test MAE)

Sample predictions (XGBoost):


,Actual ($),Predicted ($),Error ($)
0,209.23,211.080002,-1.85
1,189.07,188.029999,1.04
2,138.96,139.919998,-0.96
3,121.40,107.870003,13.53
4,116.97,117.260002,-0.29
5,181.11,184.690002,-3.58
6,166.21,166.389999,-0.18
7,264.01,264.950012,-0.94
8,178.49,178.869995,-0.38
9,245.07,244.699997,0.37


---
## 11. Save Model Artifacts

We save **both** pipelines + the best one as the default for `prediction_v2.py`.

In [28]:
SAVE_DIR = r"D:\FLIGHT PRICE PREDICTOR\BASE"

# Save both pipelines
joblib.dump(xgb_pipeline, f"{SAVE_DIR}\\xgb_pipeline_v2.pkl")
print("✅ Saved: xgb_pipeline_v2.pkl")

joblib.dump(lgb_pipeline, f"{SAVE_DIR}\\lgb_pipeline_v2.pkl")
print("✅ Saved: lgb_pipeline_v2.pkl")

# Save best as default for prediction_v2.py
joblib.dump(best_pipeline, f"{SAVE_DIR}\\flight_pipeline_v2.pkl")
print(f"✅ Saved: flight_pipeline_v2.pkl  (best = {best_name})")

# Route encoding + global mean + feature columns
joblib.dump(smooth_route_encoding, f"{SAVE_DIR}\\route_encoding_v2.pkl")
print("✅ Saved: route_encoding_v2.pkl")

joblib.dump(global_mean, f"{SAVE_DIR}\\global_mean_v2.pkl")
print("✅ Saved: global_mean_v2.pkl")

feature_columns = list(X_train.columns)
joblib.dump(feature_columns, f"{SAVE_DIR}\\feature_columns_v2.pkl")
print("✅ Saved: feature_columns_v2.pkl")

print(f"\n📁 All artifacts saved to: {SAVE_DIR}")
print("👉 Now use prediction_v2.py to make predictions.")

✅ Saved: xgb_pipeline_v2.pkl
✅ Saved: lgb_pipeline_v2.pkl
✅ Saved: flight_pipeline_v2.pkl  (best = XGBoost)
✅ Saved: route_encoding_v2.pkl
✅ Saved: global_mean_v2.pkl
✅ Saved: feature_columns_v2.pkl

📁 All artifacts saved to: D:\FLIGHT PRICE PREDICTOR\BASE
👉 Now use prediction_v2.py to make predictions.


---
## 12. Quick Test: Verify Saved Pipeline

In [29]:
# Load the saved best pipeline and test on a sample
loaded = joblib.load(f"{SAVE_DIR}\\flight_pipeline_v2.pkl")

sample = X_test.iloc[[0]]
pred = loaded.predict(sample)
actual = y_test.iloc[0]

print(f"Sample input:\n{sample.to_string()}")
print(f"\nActual fare:    ${actual:.2f}")
print(f"Predicted fare: ${pred[0]:.2f}")
print(f"Error:          ${actual - pred[0]:+.2f}")
print("\n✅ Pipeline works correctly!")

Sample input:
         MONTH  DAY  DAY_OF_WEEK ORIGIN_AIRPORT DESTINATION_AIRPORT  SCHEDULED_TIME  DISTANCE          AIRLINE  DEP_HOUR  ARR_HOUR  DEP_MIN  ARR_MIN  IS_WEEKEND   DEP_TIME_BIN  IS_HOLIDAY_SEASON      DISTANCE_BIN    ROUTE_te
4487499     12    1            2            PVD                 MCO           179.0      1072  JetBlue Airways        18        21       20       19           0  Evening_Night                  1  (1068.0, 2130.0]  210.716988

Actual fare:    $209.23
Predicted fare: $211.08
Error:          $-1.85

✅ Pipeline works correctly!
